# Offline cached-data execution check

Source: `tc_gpt_small_tokens_places_min_acts_codexgen.ipynb`. Layer: 0.

The initialization and source-prompt cells below come from the companion notebook. This bounded run uses CPU, one layer, all 25,600 cached sequences for retrieval, and at most two retrieved sequences for token inspection. It does not reproduce every exploratory cell or full-extent token diagnostic. Cached files are read without downloading or regenerating activations.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import torch
torch.set_num_threads(4)
torch.set_grad_enabled(False)


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [2]:
%matplotlib inline

In [3]:
# ! git pull


In [4]:
import gc
import os
import sys
from pathlib import Path

# Read the supplied caches without making Hub requests.
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')


'1'

In [5]:
def add_library_level(level=4):
    suf_path = ['..']
    path = '..'
    for i in range(0, level):
        join_path = suf_path * i
        path = '/'.join(join_path)
        module_path = os.path.abspath(os.path.join(path))
        if module_path not in sys.path:
            sys.path.append(module_path)
            print(f'Appendeding {path}')


In [6]:
add_library_level(level=5)


Appendeding 
Appendeding ..
Appendeding ../..
Appendeding ../../..
Appendeding ../../../..


In [7]:
import multiprocessing


In [8]:
from src.lattmc.utils import init_device, empty_cache


In [9]:
from src.lattmc.fca.utils import *
from src.lattmc.fca.fca_utils_codexgen import *


In [10]:
from src.lattmc.tc.transcoder_analyzers_codexgen import (
    ConceptAnalysis, init_analyzer,
)


In [11]:
from src.lattmc.sae.nlp_sae_utils import gen_concept


In [12]:
import logging


In [13]:
workers = multiprocessing.cpu_count()
workers


14

In [14]:
SEED = 2026


In [15]:
logging.basicConfig(level=logging.INFO)


In [16]:
device = torch.device('cpu')
device

device(type='cpu')

In [17]:
torch.__version__


'2.14.0'

In [18]:
np.__version__


'2.5.3'

In [19]:
# np.set_printoptions(precision=4, suppress=True)


In [20]:
PATH = Path('data')
GPT2 = PATH / 'transcoders' / 'gpt2'
OWT_TOKENS_DIR = GPT2 / 'owt_tokens'
TOKENS_PATH = OWT_TOKENS_DIR / 'owt_tokens_torch.pt'
OWT_TOKENS_DIR.mkdir(exist_ok=True, parents=True)


In [21]:
! ls {GPT2}


V0.npz                   v_max0.npz               v_min4.npz
V1.npz                   v_max1.npz               v_min5.npz
V10.npz                  v_max10.npz              v_min6.npz
V11.npz                  v_max11.npz              v_min7.npz
V2.npz                   v_max2.npz               v_min8.npz
V3.npz                   v_max3.npz               v_min9.npz
V4.npz                   v_max4.npz               v_min_nonzero0.npz
V5.npz                   v_max5.npz               v_min_nonzero1.npz
V6.npz                   v_max6.npz               v_min_nonzero10.npz
V7.npz                   v_max7.npz               v_min_nonzero11.npz
V8.npz                   v_max8.npz               v_min_nonzero2.npz
V9.npz                   v_max9.npz               v_min_nonzero3.npz
layer_error              v_min0.npz               v_min_nonzero4.npz
owt_tokens               v_min1.npz               v_min_nonzero5.npz
owt_tokens_canonical     v_min10.npz              v_min_nonzero6.npz
owt_tokens_

In [22]:
GPT2.absolute()


PosixPath('/Users/ltsinadze/git/my_papers/notebooks/transcoders/data/transcoders/gpt2')

In [23]:
! ls {TOKENS_PATH}


data/transcoders/gpt2/owt_tokens/owt_tokens_torch.pt


In [24]:
! ls {str(TOKENS_PATH.absolute())}


/Users/ltsinadze/git/my_papers/notebooks/transcoders/data/transcoders/gpt2/owt_tokens/owt_tokens_torch.pt


In [25]:
layers = list(range(12))
layers = [0]
# layers = list(range(12))
# layers

# Fail early instead of silently rebuilding a missing cache.
cache_files = [TOKENS_PATH]
for layer_id in layers:
    for prefix in ['V', 'v_min', 'v_max', 'v_min_nonzero']:
        cache_files.append(GPT2 / f'{prefix}{layer_id}.npz')
missing = [str(path) for path in cache_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing caches: ' + ', '.join(missing))


In [26]:
tr_analyzer = init_analyzer(
    layers,
    TOKENS_PATH,
    GPT2,
    model_name='gpt2-small',
    device=device,
    tr_or_sae=True,
    vector_dir=GPT2,
)


INFO:src.lattmc.tc.transcoder_utils:Loading transcoder for layer 0


INFO:src.lattmc.tc.transcoder_utils:Loaded file final_sparse_autoencoder_gpt2-small_blocks.0.ln2.hook_normalized_24576.pt


INFO:src.lattmc.tc.transcoder_utils:Base GPT-2 and transcoder True layer=0 weights loaded successfully.


INFO:src.lattmc.tc.transcoder_utils:Transcoder for layer 0 loaded


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

INFO:src.lattmc.tc.transcoder_fca_codexgen:data/transcoders/gpt2/v_min_nonzero0.npz exists


INFO:src.lattmc.tc.transcoder_fca_codexgen:data/transcoders/gpt2/v_min_nonzero0.npz loaded


INFO:src.lattmc.tc.transcoder_fca_codexgen:data/transcoders/gpt2/v_min0.npz exists


INFO:src.lattmc.tc.transcoder_fca_codexgen:data/transcoders/gpt2/v_max0.npz exists


INFO:src.lattmc.tc.transcoder_fca_codexgen:data/transcoders/gpt2/v_min_nonzero0.npz exists


In [27]:
empty_cache()


0

In [28]:
tr_analyzer.transcoder.model.cfg.device


'cpu'

In [29]:
empty_cache()


0

In [30]:
{i: f.V.shape for i, f in tr_analyzer.fcas.items()}

{0: (25600, 24576)}

In [31]:
tr_analyzer.tokens.shape


torch.Size([25600, 128])

In [32]:
tr_analyzer.tokens[0][0]


tensor(50256)

In [33]:
tn = to_numpy(tr_analyzer.tokens[0][0])
tn.item()


50256

In [34]:
text = ' New York City'
text


' New York City'

In [35]:
text_idx = 3457
text = tr_analyzer.texts[text_idx]
prompt = tr_analyzer.corpus[text_idx]
text


' New York City, Stuart Kitchen and Alexander Haddow. They waited until September 1952 to publish their findings, which appeared in back-to-back papers in the Transactions of the Royal Society of Tropical Medicine and Hygiene. In addition to experiments with more rhesus monkeys and mice, their exhaustive studies put the Zika virus into grivet and red tail monkeys, cotton rats, guinea pigs, and rabbits. The virus caused damage to neurons only in mice. Six of 99 humans they tested in four different Ugandan locales had antibodies to their new bug, but none showed evidence of disease.\n\nAs one of the two'

In [36]:
concept_an = ConceptAnalysis(prompt, tr_analyzer)


In [37]:
concept_an.corpus[3457]


tensor([50256,   968,  1971,  2254,    11, 22559, 22541,   290, 10009,   367,
         2860,   322,    13,  1119, 13488,  1566,  2693, 26352,   284,  7715,
          511,  6373,    11,   543,  4120,   287,   736,    12,  1462,    12,
         1891,  9473,   287,   262, 46192,   286,   262,  8111,  7023,   286,
        37956, 11558,   290,  6707, 28363,    13,   554,  3090,   284, 10256,
          351,   517,   374,   956,   385, 26697,   290, 10693,    11,   511,
        36049,  3640,  1234,   262, 30861,  9471,   656, 11120, 16809,   290,
         2266,  7894, 26697,    11, 15985, 13623,    11,   915, 18343, 22333,
           11,   290, 33043,    13,   383,  9471,  4073,  2465,   284, 16890,
          691,   287, 10693,    13,  9699,   286,  7388,  5384,   484,  6789,
          287,  1440,  1180, 24384, 42509,  1957,   274,   550, 30869,   284,
          511,   649,  5434,    11,   475,  4844,  3751,  2370,   286,  4369,
           13,   198,   198,  1722,   530,   286,   262,   734])

In [38]:
concept_an.analyze_concepts()


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:Text tokens for prompt: <|endoftext|> New York City, Stuart Kitchen and Alexander Haddow. They waited until September 1952 to publish their findings, which appeared in back-to-back papers in the Transactions of the Royal Society of Tropical Medicine and Hygiene. In addition to experiments with more rhesus monkeys and mice, their exhaustive studies put the Zika virus into grivet and red tail monkeys, cotton rats, guinea pigs, and rabbits. The virus caused damage to neurons only in mice. Six of 99 humans they tested in four different Ugandan locales had antibodies to their new bug, but none showed evidence of disease.

As one of the two


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:0 50256: <|endoftext|>


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:1 968:  New


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:2 1971:  York


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:3 2254:  City


...

INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:123 1722: As


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:124 530:  one


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:125 286:  of


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:126 262:  the


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:127 734:  two


In [39]:
layer = 0

In [40]:
import json
import importlib.metadata as metadata

versions = {
    name: metadata.version(name)
    for name in ['sae-lens', 'transformer-lens', 'transformers', 'torch']
}
print(versions)
print('tokens:', tuple(tr_analyzer.tokens.shape))
print('matrix:', tr_analyzer.fcas[layer].V.shape)
rows = [3457, 5411, 117, 21481, 1924, 4042]
checks = []
positions = {
    3457: [1, 2, 3], 5411: [15, 16, 17], 117: [30, 31],
    21481: [24, 25, 26], 1924: [15, 39, 94], 4042: [8, 82],
}
for row in rows:
    fresh = tr_analyzer.tr_utils.run_transcoders(
        tr_analyzer.corpus[row], [layer],
    )[layer]
    pooled = fresh.max(axis=0)
    cached = tr_analyzer.fcas[layer].V[row]
    changed = np.flatnonzero((pooled > 0) != (cached > 0))
    result = {
        'row': row,
        'max_abs_error': float(np.max(np.abs(pooled - cached))),
        'close': bool(np.allclose(pooled, cached, atol=1e-3, rtol=1e-3)),
        'support_disagreements': int(np.sum((pooled > 0) != (cached > 0))),
        'support_changes': [
            dict(feature=int(i), fresh=float(pooled[i]),
                 cached=float(cached[i])) for i in changed
        ],
    }
    checks.append(result)
    print(result)
assert all(item['close'] for item in checks), 'Activation cache mismatch'

{'sae-lens': '6.51.1', 'transformer-lens': '3.9.0', 'transformers': '5.17.0', 'torch': '2.14.0'}
tokens: (25600, 128)
matrix: (25600, 24576)
{'row': 3457, 'max_abs_error': 1.9073486328125e-05, 'close': True, 'support_disagreements': 0, 'support_changes': []}
{'row': 5411, 'max_abs_error': 2.2172927856445312e-05, 'close': True, 'support_disagreements': 0, 'support_changes': []}


{'row': 117, 'max_abs_error': 3.4332275390625e-05, 'close': True, 'support_disagreements': 0, 'support_changes': []}
{'row': 21481, 'max_abs_error': 1.71661376953125e-05, 'close': True, 'support_disagreements': 0, 'support_changes': []}
{'row': 1924, 'max_abs_error': 1.52587890625e-05, 'close': True, 'support_disagreements': 0, 'support_changes': []}


{'row': 4042, 'max_abs_error': 2.288818359375e-05, 'close': True, 'support_disagreements': 0, 'support_changes': []}


In [41]:
# Validate all cached bounds and support floors without a second dense copy.
fca = tr_analyzer.fcas[layer]
matrix = fca.V
lower = np.full(matrix.shape[1], np.inf, dtype=matrix.dtype)
upper = np.zeros(matrix.shape[1], dtype=matrix.dtype)
floor = lower.copy()
for start in range(0, len(matrix), 256):
    block = matrix[start:start + 256]
    assert np.all(np.isfinite(block)) and np.all(block >= 0)
    lower = np.minimum(lower, block.min(axis=0))
    upper = np.maximum(upper, block.max(axis=0))
    positive_min = np.min(block, axis=0, where=block > 0, initial=np.inf)
    floor = np.minimum(floor, positive_min)
floor[np.isinf(floor)] = 0
assert np.array_equal(lower, fca.v_min.ravel())
assert np.array_equal(upper, fca.v_max.ravel())
assert np.array_equal(floor, fca.v_min_nonzeros.ravel())
print('All cached minima, maxima, and positive support floors: PASS')

All cached minima, maxima, and positive support floors: PASS


In [42]:
# Exercise the source positions for every empirical example.
queries = []
for row in rows:
    analysis = ConceptAnalysis(tr_analyzer.corpus[row], tr_analyzer)
    analysis.analyze_concepts()
    analysis.gen_text(
        positions[row], layer, limit=2, min_vals=True, full_tokens=False,
    )
    row_extent = analysis.c_is[layer].A
    query = np.maximum.reduce(list(analysis.v_is[layer].values()))
    closed = fca.F(row_extent)
    assert row in row_extent
    assert np.all(closed >= query)
    assert np.array_equal(fca.G(closed), row_extent)
    queries.append(dict(row=row, positions=positions[row],
                        extent_size=len(row_extent)))
print(queries)

INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:Text tokens for prompt: <|endoftext|> New York City, Stuart Kitchen and Alexander Haddow. They waited until September 1952 to publish their findings, which appeared in back-to-back papers in the Transactions of the Royal Society of Tropical Medicine and Hygiene. In addition to experiments with more rhesus monkeys and mice, their exhaustive studies put the Zika virus into grivet and red tail monkeys, cotton rats, guinea pigs, and rabbits. The virus caused damage to neurons only in mice. Six of 99 humans they tested in four different Ugandan locales had antibodies to their new bug, but none showed evidence of disease.

As one of the two


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:0 50256: <|endoftext|>


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:1 968:  New


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:2 1971:  York


INFO:/Users/ltsinadze/git/my_papers/src/lattmc/tc/tokenization_utils.py:3 2254:  City


...

4042 <|endoftext|> Bob Snow (Murder of a Cat) are set to put a modern spin on the story based on Mike Weber''s version. So, maybe it will be a very touching tale of courage in the face of little tiny people who just don''t understand. Just please, please don''t make it a talking ape. PLEASE.

Source: DeadlineCLOSE The city's newest dog park, located downtown in E. Peck Greene Park, opened its fences July 18 with a ribbon-cutting ceremony. Hali Tauxe/Democrat

Buy Photo E. Peck Greene Park, located between Duval
('ĠCat',) ('Ġdog',) [8] [82]

8992 <|endoftext|> the incident was being treated as an investigation and therefore could not comment further.

"We are really disappointed that a member of the public has acted in this way. It's a sad ending for a seal pup to be treated that way."

Luecht said animal harassment was becoming more frequent, with fur seals, penguins, sea lions and sea gulls subject to ongoing mistreatment.

"People aren't giving them the space and respect they need," 

  0%|          | 0/25600 [00:00<?, ?it/s]

 55%|█████▌    | 14150/25600 [00:00<00:00, 141496.09it/s]

100%|██████████| 25600/25600 [00:00<00:00, 141443.55it/s]

[{'row': 3457, 'positions': [1, 2, 3], 'extent_size': 47}, {'row': 5411, 'positions': [15, 16, 17], 'extent_size': 12}, {'row': 117, 'positions': [30, 31], 'extent_size': 9}, {'row': 21481, 'positions': [24, 25, 26], 'extent_size': 5}, {'row': 1924, 'positions': [15, 39, 94], 'extent_size': 41}, {'row': 4042, 'positions': [8, 82], 'extent_size': 5}]


In [43]:
# Same support-floor NYC query as the source notebook, bounded inspection.
dets, vs = concept_an.gen_text(
    [1, 2, 3], layer, limit=2, min_vals=True, full_tokens=False,
)
extent = concept_an.c_is[layer].A
print('NYC joint extent:', len(extent))
print('First row indices:', extent[:10].tolist())
assert 3457 in extent, 'The source sequence must satisfy its floor probes.'
query = np.maximum.reduce(list(concept_an.v_is[layer].values()))
fca = tr_analyzer.fcas[layer]
closed = fca.F(extent)
assert np.array_equal(fca.G(closed), extent)
assert np.all(closed >= query)
print('Sequence-level closure and extent invariance: PASS')

INFO:src.lattmc.tc.transcoder_analyzers_codexgen:idxs[i]=array([10265])


INFO:src.lattmc.tc.transcoder_analyzers_codexgen:min_val_i=np.float32(0.0024980605)


INFO:src.lattmc.tc.transcoder_analyzers_codexgen:[11.574517]


INFO:src.lattmc.tc.transcoder_analyzers_codexgen:[11.]


INFO:src.lattmc.tc.transcoder_analyzers_codexgen:i_idx=0, i=1, red_val=0.0 set_val=None vals[i]=array([11.], dtype=float32)


...

1116 <|endoftext|> accused of conspiracy to provide material support and attempting to provide material support to a foreign terrorist organization. The complaint says the men planned to reach Syria by flying to nearby countries from Minneapolis, San Diego or New York City, and lied to federal investigators when they were stopped.

Four of the men were arrested Sunday in Minneapolis and two were arrested in San Diego.

Charged are Mohamed Abdihamid Farah, 21; Adnan Abdihamid Farah, 19; Abdurahman Yasin Daud, 21; Zacharia Yusuf Abdurahman, 19; Hanad Mustafe Musse,
('ĠNew',) ('ĠYork',) ('ĠCity',) [40] [41] [42]

num_detects={1: 2, 2: 2, 3: 2}
NYC joint extent: 47
First row indices: [1019, 1116, 1348, 2699, 2964, 3021, 3032, 3052, 3053, 3240]


  0%|          | 0/25600 [00:00<?, ?it/s]

 55%|█████▍    | 14007/25600 [00:00<00:00, 140058.78it/s]

100%|██████████| 25600/25600 [00:00<00:00, 138482.47it/s]

Sequence-level closure and extent invariance: PASS


In [ ]:
report_path = (project / 'data/activation_studies/legacy'
               / 'lattconference/cached_notebook_checks'
               / 'tc_layer0.json')
report_path.parent.mkdir(parents=True, exist_ok=True)
report = dict(versions=versions, checks=checks, queries=queries,
              layer=layer, cached_bounds_verified=True,
              tokens_shape=list(tr_analyzer.tokens.shape),
              matrix_shape=list(fca.V.shape),
              nyc_extent=len(extent), offline=True,
              full_notebook_execution=False)
report_path.write_text(json.dumps(report, indent=2))
report
